In [1]:
# Install necessary libraries
!pip install transformers tqdm


In [2]:
# Import required modules
import pandas as pd
from transformers import BartTokenizer, BartForConditionalGeneration
import torch
from tqdm import tqdm
from google.colab import files

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/DTSC 5082 Datasets/cleaned_discharge.csv")
text_column = "text"  # Replace with your actual column name
df = df.dropna(subset=[text_column]).head(1000)  # Limit to first 2000 non-null entries

In [5]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# Load BART model and tokenizer
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

In [7]:
# Detect text column
text_column = next((col for col in ['text', 'abstract', 'report_text', 'clinical_summary', 'discharge_note'] if col in df.columns), None)

if not text_column:
    raise ValueError(f"❌ No valid text column found. Columns available: {df.columns.tolist()}")

print(f"✅ Using column: '{text_column}' for summarization...")

✅ Using column: 'text' for summarization...


In [8]:
# Define summarization function
def summarize_text(text, max_length=200):
    if pd.isna(text) or text.strip() == "":
        return "No text available for summarization."
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(device)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [9]:
for i in range(10):
    print(f"\n======================= ENTRY {i + 1} =======================")

    original_text = df[text_column].iloc[i]
    print("\n🔹 Original Text:\n", original_text[:1000], "...")  # Show only first 1000 chars for brevity

    summary = summarize_text(original_text)
    print("\n🔸 Summary:\n", summary)


======================= ENTRY 1 =======================

🔹 Original Text:
 Name: ** Unit No: ** Admission Date: ** Discharge Date: ** Date of Birth: ** Sex: F Service: MEDICINE Allergies: No Known Allergies / Adverse Drug Reactions Attending: ** Chief Complaint: Worsening ABD distension and pain Major Surgical or Invasive Procedure: Paracentesis History of Present Illness: ** Hepatitis C Virus cirrhosis c/b ascites, hiv on Antiretroviral Therapy,History OfIVDU, Chronic Obstructive Pulmonary Disease, bioplar, Post-Traumatic Stress Disorder, presented from Left EyeH ED with worsening abd distension over past week. Patientreports self-discontinuing lasix and spirnolactone ** weeks ago, because she feels like they don't do anything and that she doesn't want to put more chemicals in her. She does not follow Na-restricted diets. In the past week, she notes that she has been having worsening abd distension and discomfort. She denies ** edema, or Shortness of Breath, or orthopnea. She denies 

In [11]:
# Apply summarization with progress bar
tqdm.pandas()
df['summary'] = df[text_column].progress_apply(summarize_text)

100%|██████████| 1000/1000 [45:04<00:00,  2.70s/it]


In [21]:
from google.colab import files
df.to_csv("summarized_1000_clinical_data.csv", index=False)
files.download("summarized_1000_clinical_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# Preview a few rows
print(df[['text', 'summary']].head(5))

                                                text  \
0  Name: ** Unit No: ** Admission Date: ** Discha...   
1  Name: ** Unit No: ** Admission Date: ** Discha...   
2  Name: ** Unit No: ** Admission Date: ** Discha...   
3  Name: ** Unit No: ** Admission Date: ** Discha...   
4  Name: ** Unit No: ** Admission Date: ** Discha...   

                                             summary  
0  Name: ** Unit No: ** Admission Date: ** Discha...  
1  Name: ** Unit No: ** Discharge Date: ** Date o...  
2  Name: ** Unit No: ** Surgical Unit: ** Gender:...  
3  Name: ** Unit No: ** Admission Date: ** Discha...  
4  Name: ** Unit No: ** Discharge Date: ** Date o...  
